# WaterLab digital twin control walkthrough

## Goal

Inspect the virtual plant, review valves and process checkpoints, run a repeatable experiment, and submit a bounded supervisory setpoint through the same safety gate used by the dashboard. This notebook never connects to OPC UA and never writes a raw actuator.

## Setup

Start the Compose stack first. Keep `RUN_DEMO` false for read-only inspection. Set it to true only when you want this notebook to create and operate a simulated run.

In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = os.getenv("WATERLAB_BASE_URL", "http://127.0.0.1:18780/api/v1").rstrip("/")
RUN_DEMO = False

def api(path, method="GET", payload=None):
    body = json.dumps(payload).encode() if payload is not None else None
    request = Request(f"{BASE_URL}{path}", data=body, method=method)
    request.add_header("Content-Type", "application/json")
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


## Steps

### 1. Inspect the live digital twin

In [ ]:
state = api("/state")
plant = state["plant"]
print("Simulation time:", plant["simulation_time"])
print("Safety state:", plant["safety_state"])
print("Hydraulic model:", plant["note"])
print("Ollama:", state["ollama"])


### 2. Review checkpoints and valve feedback

The command is the PLC output. The position is the simulated field feedback. They can differ while a valve is travelling. Pressure loss is reported across each valve.

In [ ]:
for checkpoint in plant["checkpoints"]:
    print(f"{checkpoint['name']}: {checkpoint['status']} | {checkpoint['location']}")

print()
for name, valve in plant["valves"].items():
    print(
        f"{name:24s} {valve['status']:11s} "
        f"command={valve['command_pct']:5.1f}% position={valve['position_pct']:5.1f}% "
        f"dP={valve['differential_pressure_kpa']:6.2f} kPa flow={valve['flow_m3h']:6.1f} m3/h"
    )


### 3. Create a gated automatic experiment

This cell is disabled by default. The AI can propose pressure, quality, storage, and valve targets. The supervisor submits that structured proposal to the PLC safety gate. Only accepted setpoints receive a five-minute lease. The PLC then converts them into rate-limited actuator commands.

In [ ]:
run_id = None
if RUN_DEMO:
    run_config = {
        "scenario": "normal_day",
        "seed": 42,
        "duration_hours": 1,
        "speed": 10,
        "controller_mode": "gated_auto",
        "model": "qwen3:8b",
        "ai_decision_interval_minutes": 5,
        "memory_enabled": True,
        "memory_window": 4,
    }
    created = api("/runs", method="POST", payload=run_config)
    run_id = created["id"]
    api(f"/runs/{run_id}/start", method="POST")
    api(f"/runs/{run_id}/pause", method="POST")
    api(f"/runs/{run_id}/step", method="POST", payload={"minutes": 5})
    print("Created and stepped run:", run_id)
else:
    print("Read-only mode. Set RUN_DEMO = True to create a simulated run.")


### 4. Submit a confirmed supervisory change

This is a target request, not a direct valve write. Try changing one zone target at a time. The gate rejects unsafe ranges, large steps, low-pressure restrictions, stale sensors, and conflicting valve actions.

In [ ]:
if RUN_DEMO and run_id:
    manual_request = {
        "changes": {
            "pressure_target_m": 45,
            "zone_1_isolation_target_pct": 90,
        },
        "confirmation": "CONFIRM",
    }
    gate_result = api("/control/manual", method="POST", payload=manual_request)
    print(json.dumps(gate_result, indent=2))


## Checks

Read the memory endpoint and confirm that each retrieved episode came from an audited decision. Memory is sent back to the local model as bounded context. It cannot override the deterministic gate.

In [ ]:
memory = api("/memory")
print(json.dumps(memory, indent=2))

if run_id:
    print(json.dumps(api(f"/runs/{run_id}/metrics"), indent=2))


## Next Steps

Run the same seed in baseline, shadow, and gated automatic modes. Compare demand served, pressure error, energy, chemical use, accepted proposals, and safety violations. Use the unsafe AI scenario to show that a rejected proposal never reaches an OPC UA actuator node.